# Funciones

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import utils as ut
from datasets import load_dataset
import pandas as pd
import os
import pickle
import time

### MF - DFA

In [113]:
import numpy as np
from numba import njit, prange
import utils as ut

# =============================================================================
# Partición en ventanas desde las 4 esquinas
# =============================================================================

@njit
def idxy_4(img_shape, s):

    nx, ny = img_shape
    rx = int((nx % s) != 0)  # 1 si hay residuo en filas
    ry = int((ny % s) != 0)  # 1 si hay residuo en columnas

    # Número total de ventanas considerando las 4 orientaciones
    n = (nx // s) * (ny // s) * (1 + rx + ry + (rx * ry))
 
    idxy = np.zeros((n, 2), dtype=np.int32)
    idx = 0

    # Esquina superior izquierda → (arriba-abajo, izquierda-derecha)
    for i in range(0, nx - (s * rx), s):
        for j in range(0, ny - (s * ry), s):
            idxy[idx, 0] = i
            idxy[idx, 1] = j
            idx += 1

    # Esquina superior derecha → (arriba-abajo, derecha-izquierda)
    if ry != 0:
        for i in range(0, nx - (s * rx), s):
            for j in range(ny - (s * ry), 0, -s):
                idxy[idx, 0] = i
                idxy[idx, 1] = j
                idx += 1

    # Esquina inferior izquierda → (abajo-arriba, izquierda-derecha)
    if rx != 0:
        for i in range(nx - (s * rx), 0, -s):
            for j in range(0, ny - (s * ry), s):
                idxy[idx, 0] = i
                idxy[idx, 1] = j
                idx += 1

        # Esquina inferior derecha → (abajo-arriba, derecha-izquierda)
        if ry != 0:
            for i in range(nx - (s * rx), 0, -s):
                for j in range(ny - (s * ry), 0, -s):
                    idxy[idx, 0] = i
                    idxy[idx, 1] = j
                    idx += 1

    return idxy


@njit(fastmath=True)
def poly2d_fluctuation_order1(img, i0, j0, s, integration=True):
    """MF-DFA1: detrending lineal (3 coeficientes: i, j, 1)."""
    n = s * s
    Y = np.zeros((s, s), dtype=np.float64)

    if integration:
        mean_val = 0.0
        for di in range(s):
            for dj in range(s):
                mean_val += img[i0 + di, j0 + dj]
        mean_val /= n

        for di in range(s):
            for dj in range(s):
                val = img[i0 + di, j0 + dj] - mean_val
                Y[di, dj] = val
                if di > 0:
                    Y[di, dj] += Y[di - 1, dj]
                if dj > 0:
                    Y[di, dj] += Y[di, dj - 1]
                if di > 0 and dj > 0:
                    Y[di, dj] -= Y[di - 1, dj - 1]
    else:
        for di in range(s):
            for dj in range(s):
                Y[di, dj] = img[i0 + di, j0 + dj]

    # 3 coeficientes: a*i + b*j + c
    A = np.zeros((3, 3), dtype=np.float64)
    b = np.zeros(3, dtype=np.float64)

    for di in range(s):
        for dj in range(s):
            z = Y[di, dj]
            x0 = float(di)
            x1 = float(dj)

            A[0, 0] += x0 * x0
            A[0, 1] += x0 * x1
            A[0, 2] += x0
            A[1, 1] += x1 * x1
            A[1, 2] += x1
            A[2, 2] += 1.0

            b[0] += x0 * z
            b[1] += x1 * z
            b[2] += z

    for i in range(3):
        for j in range(i):
            A[i, j] = A[j, i]

    for k in range(3):
        piv = A[k, k]
        if piv == 0.0:
            return 0.0
        inv = 1.0 / piv
        for j in range(k, 3):
            A[k, j] *= inv
        b[k] *= inv
        for i in range(3):
            if i != k:
                f = A[i, k]
                for j in range(k, 3):
                    A[i, j] -= f * A[k, j]
                b[i] -= f * b[k]

    res = 0.0
    for di in range(s):
        for dj in range(s):
            z_hat = b[0] * di + b[1] * dj + b[2]
            d = Y[di, dj] - z_hat
            res += d * d

    return np.sqrt(res / n)


@njit(fastmath=True)
def poly2d_fluctuation_order2(img, i0, j0, s, integration=True):

    n = s * s

    # --- Paso 1: Integración local (suma acumulada dentro de la ventana) ---
    Y = np.zeros((s, s), dtype=np.float64)

    if integration:
        # Sustraer media local
        mean_val = 0.0
        for di in range(s):
            for dj in range(s):
                mean_val += img[i0 + di, j0 + dj]
        mean_val /= n

        # Suma acumulada 2D local
        for di in range(s):
            for dj in range(s):
                val = img[i0 + di, j0 + dj] - mean_val
                Y[di, dj] = val
                if di > 0:
                    Y[di, dj] += Y[di - 1, dj]
                if dj > 0:
                    Y[di, dj] += Y[di, dj - 1]
                if di > 0 and dj > 0:
                    Y[di, dj] -= Y[di - 1, dj - 1]
    else:
        # Sin integración: usar valores directos
        for di in range(s):
            for dj in range(s):
                Y[di, dj] = img[i0 + di, j0 + dj]

    # --- Paso 2: Detrending polinomial de segundo orden ---
    A = np.zeros((6, 6), dtype=np.float64)
    b = np.zeros(6, dtype=np.float64)

    for di in range(s):
        for dj in range(s):
            z = Y[di, dj]

            # xc = di - (s - 1) / 2.0
            # yc = dj - (s - 1) / 2.0
            # x0 = xc
            # x1 = yc

            x0 = di * di
            x1 = dj * dj
            x2 = di * dj
            x3 = di
            x4 = dj

            A[0, 0] += x0 * x0
            A[0, 1] += x0 * x1
            A[0, 2] += x0 * x2
            A[0, 3] += x0 * x3
            A[0, 4] += x0 * x4
            A[0, 5] += x0

            A[1, 1] += x1 * x1
            A[1, 2] += x1 * x2
            A[1, 3] += x1 * x3
            A[1, 4] += x1 * x4
            A[1, 5] += x1

            A[2, 2] += x2 * x2
            A[2, 3] += x2 * x3
            A[2, 4] += x2 * x4
            A[2, 5] += x2

            A[3, 3] += x3 * x3
            A[3, 4] += x3 * x4
            A[3, 5] += x3

            A[4, 4] += x4 * x4
            A[4, 5] += x4

            A[5, 5] += 1.0

            b[0] += x0 * z
            b[1] += x1 * z
            b[2] += x2 * z
            b[3] += x3 * z
            b[4] += x4 * z
            b[5] += z

    for i in range(6):
        for j in range(i):
            A[i, j] = A[j, i]

    for k in range(6):
        piv = A[k, k]
        if piv == 0.0:
            return 0.0
        inv = 1.0 / piv
        for j in range(k, 6):
            A[k, j] *= inv
        b[k] *= inv

        for i in range(6):
            if i != k:
                f = A[i, k]
                for j in range(k, 6):
                    A[i, j] -= f * A[k, j]
                b[i] -= f * b[k]

    # --- Paso 3: Residuos y RMSE ---
    res = 0.0
    for di in range(s):
        for dj in range(s):
            z_hat = (b[0] * di * di + b[1] * dj * dj + b[2] * di * dj
                     + b[3] * di + b[4] * dj + b[5])
            d = Y[di, dj] - z_hat
            res += d * d

    return np.sqrt(res / n)


@njit(fastmath=True)
def poly2d_fluctuation_order3(img, i0, j0, s, integration=True):
    """MF-DFA3: detrending cúbico (10 coeficientes)."""
    n = s * s
    Y = np.zeros((s, s), dtype=np.float64)

    if integration:
        mean_val = 0.0
        for di in range(s):
            for dj in range(s):
                mean_val += img[i0 + di, j0 + dj]
        mean_val /= n

        for di in range(s):
            for dj in range(s):
                val = img[i0 + di, j0 + dj] - mean_val
                Y[di, dj] = val
                if di > 0:
                    Y[di, dj] += Y[di - 1, dj]
                if dj > 0:
                    Y[di, dj] += Y[di, dj - 1]
                if di > 0 and dj > 0:
                    Y[di, dj] -= Y[di - 1, dj - 1]
    else:
        for di in range(s):
            for dj in range(s):
                Y[di, dj] = img[i0 + di, j0 + dj]

    # 10 coeficientes: i³, j³, i²j, ij², i², j², ij, i, j, 1
    NC = 10
    A = np.zeros((NC, NC), dtype=np.float64)
    bv = np.zeros(NC, dtype=np.float64)

    for di in range(s):
        for dj in range(s):
            z = Y[di, dj]
            ii = float(di)
            jj = float(dj)

            # Bases: i³, j³, i²j, ij², i², j², ij, i, j, 1
            x = np.zeros(NC, dtype=np.float64)
            x[0] = ii * ii * ii
            x[1] = jj * jj * jj
            x[2] = ii * ii * jj
            x[3] = ii * jj * jj
            x[4] = ii * ii
            x[5] = jj * jj
            x[6] = ii * jj
            x[7] = ii
            x[8] = jj
            x[9] = 1.0

            for a in range(NC):
                for bb in range(a, NC):
                    A[a, bb] += x[a] * x[bb]
                bv[a] += x[a] * z

    for i in range(NC):
        for j in range(i):
            A[i, j] = A[j, i]

    for k in range(NC):
        piv = A[k, k]
        if piv == 0.0:
            return 0.0
        inv = 1.0 / piv
        for j in range(k, NC):
            A[k, j] *= inv
        bv[k] *= inv
        for i in range(NC):
            if i != k:
                f = A[i, k]
                for j in range(k, NC):
                    A[i, j] -= f * A[k, j]
                bv[i] -= f * bv[k]

    res = 0.0
    for di in range(s):
        for dj in range(s):
            ii = float(di)
            jj = float(dj)
            z_hat = (bv[0] * ii * ii * ii + bv[1] * jj * jj * jj
                     + bv[2] * ii * ii * jj + bv[3] * ii * jj * jj
                     + bv[4] * ii * ii + bv[5] * jj * jj
                     + bv[6] * ii * jj + bv[7] * ii + bv[8] * jj + bv[9])
            d = Y[di, dj] - z_hat   # uv −  ̃uv
            res += d * d

    return np.sqrt(res / n) # F(u,w,s)


@njit(parallel=True, fastmath=True)
def local_fluctuation(Y, idxy, s, integration = True, degree_trend = 2):
    n = idxy.shape[0]
    F_uw = np.zeros(n)
    for k in prange(n):
        i0 = idxy[k, 0]
        j0 = idxy[k, 1]
        if degree_trend == 1:
            F_uw[k] = poly2d_fluctuation_order1(Y, i0, j0, s, integration = integration)
        if degree_trend == 2:
            F_uw[k] = poly2d_fluctuation_order2(Y, i0, j0, s, integration = integration)    
        if degree_trend == 3:
            F_uw[k] = poly2d_fluctuation_order3(Y, i0, j0, s, integration = integration)
    return F_uw


# # =============================================================================
# # Función de fluctuación generalizada Fq(s)
# # =============================================================================

@njit(parallel=True, fastmath=True)
def mf_fluctuation(f_loc, qs):
    nq = qs.shape[0]
    n = f_loc.shape[0]
    # Omitir fluctuaciones nulas (ventanas degeneradas)
    f_loc_0 = np.array([f for f in f_loc if f > 0])
    n_0 = f_loc_0.shape[0]

    Fq = np.zeros(nq)
    for iq in prange(nq):
        q = qs[iq]

        if q == 0.0:
            # Media geométrica (límite de Fq cuando q→0)
            acc = 0.0
            for i in range(n_0):
                acc += np.log(f_loc_0[i])
            Fq[iq] = np.exp(acc / n)
        else:
            acc = 0.0
            for i in range(n_0):
                acc += f_loc_0[i] ** q
            Fq[iq] = (acc / n) ** (1.0 / q) #Fqs

    return Fq


# =============================================================================
# MF-DFA completo con extracción de características
# =============================================================================

def mf_dfa_features(
    img,
    q_min=-5.0,
    q_max=5.0,
    s_min=6,
    s_max=0.1,
    integration=True,
    degree_trend = 2,
    degree_scales = 2,
):

    # ---- Valores de q ----
    # qs = np.array(ut.vals_Qs(q_n=q_min, q_p=q_max))
    qs = np.arange(q_min - 0.5,q_max + 0.75,0.25)

    # ---- Escalas ----
    img_shape = img.shape
    if type(s_min) == int:
        if type(s_max) == int:
            scales = ut.bineo(s_min, s_max, degree = degree_scales)
        else:
            scales = ut.bineo(s_min, int(min(img.shape) * s_max), degree = degree_scales)
    elif type(s_max) == int:
        scales = ut.bineo( int(min(img.shape) * s_min), s_max, degree = degree_scales)
    else:
        scales = ut.bineo( int(min(img.shape) * s_min), int(min(img.shape) * s_max), degree = degree_scales)


    Y = img.copy()
    

    nq = qs.shape[0]
    ns = scales.shape[0]

    Fqs = np.zeros((nq, ns), dtype=np.float64)
    F_s = []  # Para el Hurst clásico (q=2)

    # ---- MF-DFA: fluctuaciones por escala ----
    for is_, s in enumerate(scales):
        idxy = idxy_4(img_shape, s)
        idxy = np.ascontiguousarray(idxy, dtype=np.int32)

        f_loc = local_fluctuation(Y, idxy, int(s), integration = integration, degree_trend=degree_trend)

        # Fluctuación generalizada para todos los q
        Fqs[:, is_] = mf_fluctuation(f_loc, qs)

        # Fluctuación clásica (q=2) para el exponente de Hurst
        f_s = float(np.sqrt(np.mean(np.power(f_loc, 2))))
        F_s.append(f_s)

    # ---- Exponente de Hurst clásico ----
    vals = np.polyfit(np.log(scales), np.log(F_s), deg=1)

    # ---- h(q): exponente de Hurst generalizado ----
    log_s = np.log(scales)
    hq = np.zeros(nq, dtype=np.float64)
    for iq in range(nq):
        coeffs = np.polyfit(log_s, np.log(Fqs[iq, :]), 1)
        hq[iq] = coeffs[0]

    # ---- τ(q): función de masa ----
    D = 2.0  # Dimensión del espacio (imagen 2D)
    tau_q = qs * hq - D

    # ---- Espectro multifractal f(α) via transformada de Legendre ----
    alpha = np.gradient(tau_q, qs)       # α(q) = dτ/dq
    f_alpha = qs * alpha - tau_q         # f(α) = q·α - τ(q)

    # ---- Empaquetar datos completos ----
    data = {
        'alpha': np.array(alpha)[2:-2],
        'f_alpha': np.array(f_alpha)[2:-2],
        'hq': np.array(hq)[2:-2],
        'tq': np.array(tau_q)[2:-2],
        'qs': np.array(qs)[2:-2],
        's_sizes': np.array(scales),
        'fluctuations': Fqs[2:-2],
    }


    # ---- Extracción de 14 características ----

    # Extremos y ancho del espectro
    a_max = data['alpha'][0]          # Extremo derecho (q más negativo)
    a_min = data['alpha'][-1]         # Extremo izquierdo (q más positivo)
    dif_a = np.abs(a_max - a_min)     # Ancho total Δα

    # Posición del máximo
    a_star = data['alpha'][data['f_alpha'] == np.max(data['f_alpha'])][0]

    # Asimetría del espectro
    dif_L = np.abs(a_star - a_min)    # Brazo izquierdo
    dif_R = np.abs(a_max - a_star)    # Brazo derecho
    asy_i = (dif_L - dif_R) / (dif_L + dif_R)  # Índice de asimetría

    # Alturas del espectro
    f_max = data['f_alpha'][0]        # Altura en α_max
    f_min = data['f_alpha'][-1]       # Altura en α_min
    dif_f = np.abs(np.max(data['f_alpha']) - np.min(data['f_alpha']))

    # Ajuste cuadrático de τ(q): captura la curvatura global
    a, b, c = np.polyfit(data['qs'], data['tq'], 2)

    features = {
        'a_max': float(a_max),
        'a_min': float(a_min),
        'dif_a': float(dif_a),
        'a_star': float(a_star),
        'dif_L': float(dif_L),
        'dif_R': float(dif_R),
        'asy_i': float(asy_i),
        'f_max': float(f_max),
        'f_min': float(f_min),
        'dif_f': float(dif_f),
        'a': float(a),
        'b': float(b),
        'c': float(c),
        'Hurst': float(vals[0])
    }

    return data, features

### Dimensión generalizada

In [114]:
import numpy as np
import utils as ut
from numba import njit, prange



@njit(parallel=True, fastmath=True)
def measures_int(idxy, img, s):
    n = len(idxy)
    n_pixels = s * s

    intensity = np.zeros(n, dtype=np.float64)
    variance = np.zeros(n, dtype=np.float64)
    entropy = np.zeros(n, dtype=np.float64)

    for k in prange(n):
        x = idxy[k, 0]
        y = idxy[k, 1]

        # Un solo recorrido: suma, suma de cuadrados e histograma
        sum_val = 0.0
        sum_sq = 0.0
        counts = np.zeros(256, dtype=np.int64)

        for i in range(s):
            for j in range(s):
                val = img[x + i, y + j]
                sum_val += val
                sum_sq += val * val
                idx = int(val)
                if idx < 0:
                    idx = 0
                if idx > 255:
                    idx = 255
                counts[idx] += 1

        # Intensidad
        intensity[k] = sum_val

        # Varianza: E[X²] - E[X]²
        mean_val = sum_val / n_pixels
        variance[k] = sum_sq / n_pixels - mean_val * mean_val

        # Entropía de Shannon normalizada
        n_unique = 0
        for u in range(256):
            if counts[u] > 0:
                n_unique += 1

        if n_unique <= 1:
            entropy[k] = 0.0
        else:
            H = 0.0
            log_n = np.log(n_unique)
            for u in range(256):
                if counts[u] > 0:
                    p = counts[u] / n_pixels
                    H -= p * np.log(p)
            entropy[k] = H #/ log_n

    return intensity, variance, entropy


@njit(parallel=True, fastmath=True)
def measures(idxy, img, s):

    n = len(idxy)
    n_pixels = s * s
 
    intensity = np.zeros(n, dtype=np.float64)
    variance = np.zeros(n, dtype=np.float64)
    entropy = np.zeros(n, dtype=np.float64)
 
    for k in prange(n):
        x = idxy[k, 0]
        y = idxy[k, 1]
 
        # --- Un solo recorrido: suma, suma de cuadrados y valores ---
        sum_val = 0.0
        sum_sq = 0.0
        values = np.zeros(n_pixels, dtype=np.float64)
        idx = 0
 
        for i in range(s):
            for j in range(s):
                val = img[x + i, y + j]
                sum_val += val
                sum_sq += val * val
                values[idx] = val
                idx += 1
 
        # Intensidad
        intensity[k] = sum_val
 
        # Varianza: E[X²] - E[X]²
        mean_val = sum_val / n_pixels
        variance[k] = sum_sq / n_pixels - mean_val * mean_val
 
        # --- Entropía de Shannon ---
        # Contar valores únicos con búsqueda lineal
        unique_vals = np.zeros(n_pixels, dtype=np.float64)
        unique_counts = np.zeros(n_pixels, dtype=np.int64)
        n_unique = 0
 
        for i in range(n_pixels):
            v = values[i]
            found = False
            for u in range(n_unique):
                if unique_vals[u] == v:
                    unique_counts[u] += 1
                    found = True
                    break
            if not found:
                unique_vals[n_unique] = v
                unique_counts[n_unique] = 1
                n_unique += 1
 
        if n_unique <= 1:
            entropy[k] = 0.0
        else:
            H = 0.0
            for u in range(n_unique):
                p = unique_counts[u] / n_pixels
                H -= p * np.log(p)
            entropy[k] = H
 
    return intensity, variance, entropy

@njit
def idxy(img_shape, s):

    nx, ny = img_shape
    n = (nx // s) * (ny // s)
    coords = np.zeros((n, 2), dtype=np.int32)
    idx = 0

    for i in range(0, (nx // s) * s, s):
        for j in range(0, (ny // s) * s, s):
            coords[idx, 0] = i
            coords[idx, 1] = j
            idx += 1

    return coords


def mf_renyi_features(
    img,
    q_min=-5.0,
    q_max=5.0,
    s_min=6,
    s_max=0.1,
    img_256 = True,
    degree_scales = 2
):

    # ---- Valores de q ----
    # qs = np.array(ut.vals_Qs(q_n=q_min, q_p=q_max))
    qs = np.arange(q_min - 0.25, q_max + 0.25, 0.25)
    nq = len(qs)

    # ---- Escalas ----
    img_shape = img.shape
    if type(s_min) == int:
        if type(s_max) == int:
            scales = ut.bineo(s_min, s_max, degree = degree_scales)
        else:
            scales = ut.bineo(s_min, int(min(img.shape) * s_max), degree = degree_scales)
    elif type(s_max) == int:
        scales = ut.bineo( int(min(img.shape) * s_min), s_max, degree = degree_scales)
    else:
        scales = ut.bineo( int(min(img.shape) * s_min), int(min(img.shape) * s_max), degree = degree_scales)
    # scales = ut.bineo(s_min, int(min(img.shape) * s_max), degree=2)
    ns = len(scales)

    # Matrices de función de partición para cada métrica
    chi_sum = np.zeros((nq, ns), dtype=np.float64)
    chi_var = np.zeros((nq, ns), dtype=np.float64)
    chi_ent = np.zeros((nq, ns), dtype=np.float64)

    # ---- Calcular función de partición por escala ----
    for is_, s in enumerate(scales):
        xy = idxy(img_shape=img_shape, s=int(s))
        xy = np.ascontiguousarray(xy, dtype=np.int32)
        if img_256 == True:
            Sum, Var, Ent = measures_int(idxy=xy, img=img, s=int(s))
        else:
            Sum, Var, Ent = measures(idxy=xy, img=img, s=int(s))

        # Normalizar a probabilidad (excluir valores <= 0)
        p_sum = _normalize(Sum)
        p_var = _normalize(Var)
        p_ent = _normalize(Ent)

        # Función de partición para cada q
        chi_sum[:, is_] = _partition_function(p_sum, qs)
        chi_var[:, is_] = _partition_function(p_var, qs)
        chi_ent[:, is_] = _partition_function(p_ent, qs)


    # ---- Obtener espectros y características por métrica ----
    log_s = np.log(scales)

    data = {}
    features = {}

    for name, chi in [('sum', chi_sum), ('var', chi_var), ('ent', chi_ent)]:
        tq, Dq, alpha, f_alpha, chi = _compute_spectrum(chi, qs, log_s)

        data[name] = {
            'Dq': np.array(Dq),
            'tq': np.array(tq),
            'alpha': np.array(alpha),
            'f_alpha': np.array(f_alpha),
            'qs': np.array(qs),
            'scales': np.array(scales),
            'functions': np.array(chi)
        }

        feat = _extract_features(alpha, f_alpha, Dq, qs, tq)
        for key, val in feat.items():
            features[f'{name}_{key}'] = val

        # break

    return data, features


def _normalize(masses):

    p = np.copy(masses)
    p[p <= 0] = 0.0
    total = np.sum(p)
    if total > 0:
        p = p / total
    return p


def _partition_function(p, qs):

    nq = len(qs)
    chi = np.zeros(nq, dtype=np.float64)

    # Filtrar probabilidades positivas
    p_pos = p[p > 0]
    if len(p_pos) == 0:
        return chi

    for iq in range(nq):
        q = qs[iq]
        if abs(q - 1.0) < 1e-10:
            # q ≈ 1: usar Σ p_i * log(p_i) directamente
            # chi[iq] = np.exp(np.sum(p_pos * np.log(p_pos)))
            chi[iq] = - np.sum(p_pos * np.log(p_pos))
        else:
            chi[iq] = np.sum( p_pos ** q )

    return chi


def _compute_spectrum(chi, qs, log_s):

    nq = len(qs)
    tq = np.zeros(nq, dtype=np.float64)
    Dq = np.zeros(nq, dtype=np.float64)

    for iq in range(nq):
        pi = chi[iq, :]
        log_chi = np.log( pi + 1e-300)  # Evitar log(0)
        
        q = qs[iq]
        if abs(q - 1.0) < 1e-10: # q = 1
            # D_1: dimensión de información (límite)
            coeffs = np.polyfit(log_s, pi, 1)
            tq[iq] = 0.0  # τ(1) = 0 siempre
            Dq[iq] = - coeffs[0]  
            
        elif abs(q) < 1e-10: # q = 0
            # D_0: dimensión box-counting
            coeffs = np.polyfit(log_s, log_chi, 1)
            tq[iq] = coeffs[0]
            Dq[iq] = - tq[iq]  
            
        else:
            coeffs = np.polyfit(log_s, log_chi, 1)
            tq[iq] = coeffs[0]
            Dq[iq] = tq[iq] / (q - 1.0)

    # Espectro multifractal via Legendre
    alpha = np.gradient(tq, qs)
    f_alpha = qs * alpha - tq

    alpha = alpha[1:-1]
    f_alpha = f_alpha[1:-1]

    return tq, Dq, alpha, f_alpha, chi


def _extract_features(alpha, f_alpha, Dq, qs, tq):

    a_max = alpha[0]
    a_min = alpha[-1]
    dif_a = np.abs(a_max - a_min)

    a_star = alpha[np.argmax(f_alpha)]

    dif_L = np.abs(a_star - a_min)
    dif_R = np.abs(a_max - a_star)

    denom = dif_L + dif_R
    asy_i = (dif_L - dif_R) / denom if denom > 0 else 0.0

    f_max = f_alpha[0]
    f_min = f_alpha[-1]
    dif_f = np.abs(np.max(f_alpha) - np.min(f_alpha))

    # Dimensiones especiales: D0, D1, D2
    # Buscar los q más cercanos a 0, 1 y 2
    idx_q0 = np.argmin(np.abs(qs))
    idx_q1 = np.argmin(np.abs(qs - 1.0))
    idx_q2 = np.argmin(np.abs(qs - 2.0))

    D0 = float(Dq[idx_q0])
    D1 = float(Dq[idx_q1])
    D2 = float(Dq[idx_q2])

    features = {
        'a_max': float(a_max),
        'a_min': float(a_min),
        'dif_a': float(dif_a),
        'a_star': float(a_star),
        'dif_L': float(dif_L),
        'dif_R': float(dif_R),
        'asy_i': float(asy_i),
        'f_max': float(f_max),
        'f_min': float(f_min),
        'dif_f': float(dif_f),
        'D0': D0,
        'D1': D1,
        'D2': D2,
    }

    return features

### TDA

In [115]:
"""
TDA: Análisis Topológico de Datos para imágenes 2D.

Este módulo implementa el cálculo de homología persistente sobre una
filtración cubical por niveles de gris, incluyendo:
- Filtración de H0 (componentes conexas) con 8-conectividad
- Filtración de H1 (ciclos) via dualidad de Alexander con 4-conectividad
- Extracción de descriptores topológicos: entropía de persistencia,
  estadísticas de tiempos de vida y conteo de características

La implementación usa Union-Find con compresión de caminos, compilado
con numba para rendimiento en imágenes de alta resolución.

Referencia principal:
    - Avilés-Rodríguez et al. (2021). Topological Data Analysis for
      Eye Fundus Image Quality Assessment.
    - Edelsbrunner & Harer (2010). Computational Topology.
"""

import numpy as np
from numba import njit


# =============================================================================
# Union-Find con compresión de caminos
# =============================================================================

@njit
def find(parent, x):
    """
    Encuentra la raíz del componente al que pertenece x,
    con compresión de caminos para eficiencia amortizada O(α(n)).

    Parameters
    ----------
    parent : ndarray
        Arreglo de padres de la estructura Union-Find.
    x : int
        Índice del elemento a buscar.

    Returns
    -------
    int
        Raíz del componente.
    """
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # Compresión de caminos
        x = parent[x]
    return x


@njit
def union(parent, birth, death, a, b, level):
    """
    Fusiona dos componentes aplicando la regla del más viejo.

    El componente con nacimiento más temprano (más viejo) sobrevive,
    y el más joven muere en el nivel de filtración actual. Esto
    garantiza que el tiempo de vida refleje la persistencia real.

    Parameters
    ----------
    parent : ndarray
        Arreglo de padres.
    birth : ndarray
        Nivel de nacimiento de cada componente.
    death : ndarray
        Nivel de muerte de cada componente (-1 si aún vivo).
    a, b : int
        Índices de los elementos a fusionar.
    level : int
        Nivel de filtración actual (momento de la muerte).
    """
    ra = find(parent, a)
    rb = find(parent, b)

    if ra == rb:
        return

    if birth[ra] <= birth[rb]:
        parent[rb] = ra
        death[rb] = level
    else:
        parent[ra] = rb
        death[ra] = level


# =============================================================================
# Descriptores topológicos
# =============================================================================

@njit
def entropy_persistence(lifetimes):
    """
    Calcula la entropía de persistencia normalizada.

    E = -Σ p_i · log(p_i) / log(n)

    donde p_i = l_i / Σl_j es la proporción del tiempo de vida
    de la i-ésima característica. La normalización por log(n) acota
    el valor entre 0 y 1.

    Un valor bajo indica que pocas estructuras dominan el diagrama
    (imagen con organización simple). Un valor alto indica una
    distribución uniforme de tiempos de vida (complejidad en
    múltiples escalas).

    Parameters
    ----------
    lifetimes : ndarray
        Tiempos de vida (d_i - b_i) de cada característica.

    Returns
    -------
    float
        Entropía de persistencia normalizada en [0, 1].
    """
    total = 0.0
    n = 0
    for i in range(lifetimes.shape[0]):
        if lifetimes[i] > 0:
            total += lifetimes[i]
            n += 1

    if total == 0.0 or n <= 1:
        return 0.0

    H = 0.0
    for i in range(lifetimes.shape[0]):
        li = lifetimes[i]
        if li > 0:
            p = li / total
            H -= p * np.log(p)

    return H / np.log(n)


@njit
def compute_lifetime_stats(lifetimes):
    """
    Calcula estadísticas de la distribución de tiempos de vida.

    Parameters
    ----------
    lifetimes : ndarray
        Tiempos de vida (d_i - b_i) de cada característica.

    Returns
    -------
    mean_life : float
        Media de los tiempos de vida.
    std_life : float
        Desviación estándar de los tiempos de vida.
    max_life : float
        Tiempo de vida máximo (estructura más persistente).
    n_features : int
        Número de características con vida > 0.
    """
    # Filtrar tiempos de vida positivos
    n = 0
    for i in range(lifetimes.shape[0]):
        if lifetimes[i] > 0:
            n += 1

    if n == 0:
        return 0.0, 0.0, 0.0, 0

    # Media
    total = 0.0
    max_life = 0.0
    for i in range(lifetimes.shape[0]):
        li = lifetimes[i]
        if li > 0:
            total += li
            if li > max_life:
                max_life = li
    mean_life = total / n

    # Desviación estándar
    var_acc = 0.0
    for i in range(lifetimes.shape[0]):
        li = lifetimes[i]
        if li > 0:
            diff = li - mean_life
            var_acc += diff * diff
    std_life = np.sqrt(var_acc / n)

    return mean_life, std_life, max_life, n


# =============================================================================
# Imagen invertida para dualidad de Alexander
# =============================================================================

@njit
def inverse(img):
    """
    Calcula la imagen complementaria: I_inv = max(I) - I.

    Los ciclos (H1) en la imagen original corresponden a componentes
    conexas (H0) en la imagen invertida (dualidad de Alexander).

    Parameters
    ----------
    img : ndarray (H, W)
        Imagen en escala de grises.

    Returns
    -------
    new_img : ndarray (H, W)
        Imagen invertida.
    """
    img_shape = img.shape
    max_gray = np.max(img)
    new_img = np.zeros(shape=img_shape, dtype=np.int64)
    for i in range(img_shape[0]):
        for j in range(img_shape[1]):
            val = max_gray - img[i, j]
            if val < 0:
                val = -val
            new_img[i, j] = val
    return new_img


# =============================================================================
# Filtración cubical y homología persistente
# =============================================================================

@njit
def tda_fast(img, step=5):
    """
    Calcula la homología persistente H0 y H1 de una imagen mediante
    filtración cubical por niveles de gris.

    Procedimiento:
    1. Recorre los niveles de gris de menor a mayor con paso 'step'.
    2. Para H0 (componentes conexas, 8-conectividad):
       - Cada píxel sin vecinos activos NACE como nueva componente.
       - Cada píxel con vecinos se FUSIONA (el más viejo sobrevive).
    3. Para H1 (ciclos, via dualidad de Alexander):
       - Simultáneamente, aplica el mismo procedimiento sobre la
         imagen invertida con 4-conectividad.
       - Al final, invierte los tiempos (b,d) a la escala original.
    4. Extrae descriptores de ambos diagramas de persistencia.

    Parameters
    ----------
    img : ndarray (H, W)
        Imagen en escala de grises (uint8 o similar).
    step : int
        Paso de discretización de los niveles de gris.
        step=1: filtración completa (256 niveles, más lento).
        step=5: filtración gruesa (~51 niveles, más rápido).

    Returns
    -------
    features : dict-like tuple
        (H0_entropy, H0_mean, H0_std, H0_max, H0_n,
         H1_entropy, H1_mean, H1_std, H1_max, H1_n,
         H0_lifetimes, H1_lifetimes)

        Los primeros 10 valores son los descriptores escalares.
        Los últimos 2 son los arreglos de tiempos de vida completos,
        útiles para visualización de diagramas de persistencia.
    """
    H, W = img.shape
    max_gray = np.max(img)
    levels = np.arange(-step, max_gray + step, step, dtype=np.int64)
    K = len(levels) - 1
    neg_img = inverse(img)
    max_labels = H * W

    # --- Estructuras Union-Find para H0 ---
    labels = np.zeros((H + 2, W + 2), dtype=np.int64)
    parent = np.zeros(max_labels + 1, dtype=np.int64)
    birth = np.zeros(max_labels + 1, dtype=np.int64)
    death = -np.ones(max_labels + 1, dtype=np.int64)
    n_label = 0

    # --- Estructuras Union-Find para H1 (imagen invertida) ---
    # Se inicializa con un marco de borde activo (label=1) que
    # representa la componente conexa del "exterior" de la imagen.
    # Esto es necesario para la dualidad: los ciclos de la imagen
    # original se detectan como fusiones con este marco exterior.
    labels_h1 = np.zeros((H + 2, W + 2), dtype=np.int64)
    labels_h1[0, :] = 1
    labels_h1[-1, :] = 1
    labels_h1[:, 0] = 1
    labels_h1[:, -1] = 1
    parent_h1 = np.zeros(max_labels + 1, dtype=np.int64)
    parent_h1[1] = 1
    birth_h1 = np.zeros(max_labels + 1, dtype=np.int64)
    birth_h1[1] = -1  # El marco exterior nace antes de la filtración
    death_h1 = -np.ones(max_labels + 1, dtype=np.int64)
    n_label_h1 = 1

    # --- Filtración principal ---
    for k in range(K):
        level = levels[k]
        level_up = levels[k + 1]

        for i in range(H):
            for j in range(W):

                # === H0: componentes conexas (8-conectividad) ===
                if level < img[i, j] <= level_up:
                    # Buscar vecinos activos (8-conectividad)
                    # El offset +1 en labels es por el padding de borde
                    neighbors = np.zeros(8, dtype=np.int64)
                    c = 0

                    if labels[i, j] > 0:          # arriba-izquierda
                        neighbors[c] = labels[i, j]; c += 1
                    if labels[i + 1, j] > 0:       # arriba
                        neighbors[c] = labels[i + 1, j]; c += 1
                    if labels[i + 2, j] > 0:       # arriba-derecha
                        neighbors[c] = labels[i + 2, j]; c += 1
                    if labels[i, j + 1] > 0:       # izquierda
                        neighbors[c] = labels[i, j + 1]; c += 1
                    if labels[i + 2, j + 1] > 0:   # derecha
                        neighbors[c] = labels[i + 2, j + 1]; c += 1
                    if labels[i, j + 2] > 0:       # abajo-izquierda
                        neighbors[c] = labels[i, j + 2]; c += 1
                    if labels[i + 1, j + 2] > 0:   # abajo
                        neighbors[c] = labels[i + 1, j + 2]; c += 1
                    if labels[i + 2, j + 2] > 0:   # abajo-derecha
                        neighbors[c] = labels[i + 2, j + 2]; c += 1

                    if c == 0:
                        # NACE nueva componente conexa
                        n_label += 1
                        labels[i + 1, j + 1] = n_label
                        parent[n_label] = n_label
                        birth[n_label] = level_up
                    else:
                        # FUSIÓN con vecinos existentes
                        root = neighbors[0]
                        labels[i + 1, j + 1] = root
                        for t in range(1, c):
                            union(parent, birth, death,
                                  root, neighbors[t], level_up)

                # === H1: ciclos via dualidad (4-conectividad) ===
                if level - 1 <= neg_img[i, j] < level_up - 1:
                    neighbors_h1 = np.zeros(4, dtype=np.int64)
                    c_h1 = 0

                    if labels_h1[i + 1, j] > 0:       # arriba
                        neighbors_h1[c_h1] = labels_h1[i + 1, j]; c_h1 += 1
                    if labels_h1[i, j + 1] > 0:       # izquierda
                        neighbors_h1[c_h1] = labels_h1[i, j + 1]; c_h1 += 1
                    if labels_h1[i + 2, j + 1] > 0:   # derecha
                        neighbors_h1[c_h1] = labels_h1[i + 2, j + 1]; c_h1 += 1
                    if labels_h1[i + 1, j + 2] > 0:   # abajo
                        neighbors_h1[c_h1] = labels_h1[i + 1, j + 2]; c_h1 += 1

                    if c_h1 == 0:
                        # NACE nueva componente en imagen invertida
                        # (= nace un ciclo en la imagen original)
                        n_label_h1 += 1
                        labels_h1[i + 1, j + 1] = n_label_h1
                        parent_h1[n_label_h1] = n_label_h1
                        birth_h1[n_label_h1] = level_up - 1
                    else:
                        root_h1 = neighbors_h1[0]
                        labels_h1[i + 1, j + 1] = root_h1
                        for t in range(1, c_h1):
                            union(parent_h1, birth_h1, death_h1,
                                  root_h1, neighbors_h1[t], level_up - 1)

        # Compresión de etiquetas al final de cada nivel
        for i in range(H):
            for j in range(W):
                if labels[i + 1, j + 1] > 0:
                    labels[i + 1, j + 1] = find(parent, labels[i + 1, j + 1])
                if labels_h1[i + 1, j + 1] > 0:
                    labels_h1[i + 1, j + 1] = find(parent_h1, labels_h1[i + 1, j + 1])

    # =================================================================
    # Extracción de descriptores del diagrama de persistencia
    # =================================================================

    # --- H0: componentes conexas ---
    h0_birth = birth[1:n_label + 1]
    h0_death = death[1:n_label + 1]
    h0_death[h0_death == -1] = max_gray  # Componentes que nunca mueren
    h0_lifetimes = h0_death - h0_birth

    h0_entropy = entropy_persistence(h0_lifetimes)
    h0_mean, h0_std, h0_max, h0_n = compute_lifetime_stats(h0_lifetimes)

    # --- H1: ciclos (transformar de vuelta a escala original) ---
    # En la imagen invertida, birth y death están invertidos.
    # Transformación: b_original = max_gray - d_invertido + step
    #                 d_original = max_gray - b_invertido + step
    # Se excluye label=1 (marco exterior) y características con vida=0
    raw_birth_h1 = death_h1[2:n_label_h1 + 1]
    raw_death_h1 = birth_h1[2:n_label_h1 + 1]

    final_birth_list = []
    final_death_list = []
    for b_inv, d_inv in zip(raw_birth_h1, raw_death_h1):
        b_orig = max_gray - b_inv + step
        d_orig = max_gray - d_inv + step
        if b_orig != d_orig:  # Excluir características con vida nula
            final_birth_list.append(b_orig)
            final_death_list.append(d_orig)

    if len(final_birth_list) > 0:
        final_birth_h1 = np.array(final_birth_list)
        final_death_h1 = np.array(final_death_list)
        h1_lifetimes = final_death_h1 - final_birth_h1
    else:
        h1_lifetimes = np.zeros(1, dtype=np.int64)

    h1_entropy = entropy_persistence(h1_lifetimes)
    h1_mean, h1_std, h1_max, h1_n = compute_lifetime_stats(h1_lifetimes)

    return (h0_entropy, h0_mean, h0_std, h0_max, h0_n,
            h1_entropy, h1_mean, h1_std, h1_max, h1_n,
            h0_lifetimes, h1_lifetimes)


# =============================================================================
# Wrapper para extracción de características con nombres
# =============================================================================

def tda_features(img, step=5):
    """
    Extrae los 12 descriptores topológicos de una imagen, organizados
    en un diccionario con nombres descriptivos.

    Wrapper de tda_fast que devuelve los resultados en formato
    compatible con el pipeline de extracción de características.

    Parameters
    ----------
    img : ndarray (H, W)
        Imagen en escala de grises.
    step : int
        Paso de discretización de la filtración.

    Returns
    -------
    features : dict
        12 descriptores topológicos:
        - 'H0_entropy': entropía de persistencia normalizada de H0
        - 'H0_mean': media de tiempos de vida de H0
        - 'H0_std': desviación estándar de tiempos de vida de H0
        - 'H0_max': tiempo de vida máximo de H0
        - 'H0_n': número de componentes conexas
        - 'H0_norm_entropy': entropía normalizada por rango de gris
        - 'H1_entropy': entropía de persistencia normalizada de H1
        - 'H1_mean': media de tiempos de vida de H1
        - 'H1_std': desviación estándar de tiempos de vida de H1
        - 'H1_max': tiempo de vida máximo de H1
        - 'H1_n': número de ciclos
        - 'H1_norm_entropy': entropía normalizada por rango de gris
    data : dict
        Datos completos para visualización:
        - 'H0_lifetimes': tiempos de vida de H0
        - 'H1_lifetimes': tiempos de vida de H1
    """
    result = tda_fast(img.astype(np.int64), step=step)

    (h0_entropy, h0_mean, h0_std, h0_max, h0_n,
     h1_entropy, h1_mean, h1_std, h1_max, h1_n,
     h0_lifetimes, h1_lifetimes) = result

    # Normalizar media y max por el rango de gris para comparabilidad
    # entre imágenes con diferentes rangos dinámicos
    gray_range = float(np.max(img) - np.min(img))
    if gray_range == 0:
        gray_range = 1.0

    features = {
        'H0_entropy': float(h0_entropy),
        'H0_mean': float(h0_mean) / gray_range,
        'H0_std': float(h0_std) / gray_range,
        'H0_max': float(h0_max) / gray_range,
        'H0_n': int(h0_n),
        'H1_entropy': float(h1_entropy),
        'H1_mean': float(h1_mean) / gray_range,
        'H1_std': float(h1_std) / gray_range,
        'H1_max': float(h1_max) / gray_range,
        'H1_n': int(h1_n),
    }

    data = {
        'H0_lifetimes': np.array(h0_lifetimes),
        'H1_lifetimes': np.array(h1_lifetimes),
    }

    return  data,features

# Procesamiento de Caracteristicas

In [3]:
path_data = '/Users/jardigarcia/Documents/DATASETS/ART/wikiart_dataset'  # MacBook
ds = load_dataset(path_data)['train']
N = len(ds)

Resolving data files:   0%|          | 0/69 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/66 [00:00<?, ?it/s]

### Metadata

In [6]:
artistas = ds.features['artist'].names
movimientos = ds.features['style'].names
generos = ds.features['genre'].names

#### Guardar

In [24]:
dic_metadata = {'id':[],'autor':[],'movimiento':[],'genero':[]}
for i,item in enumerate(ds):
    dic_metadata['id'].append(i)
    dic_metadata['autor'].append(artistas[int(item['artist'])])
    dic_metadata['movimiento'].append(movimientos[int(item['style'])])
    dic_metadata['genero'].append(generos[int(item['genre'])])

    # if i == 10: break

df_metadata = pd.DataFrame(dic_metadata).set_index('id')
np.savez(os.path.join('DATA','metadata'), data = dic_metadata, allow_pickle=True)


#### Cargar

In [3]:
dic_metadata = np.load(os.path.join('DATA','metadata.npz'), allow_pickle=True)['data'].item()
df_metadata = pd.DataFrame(dic_metadata)
df_metadata = df_metadata[df_metadata['autor'] != 'Unknown Artist']

In [90]:
umbral = 200

conteo_artistas = df_metadata['autor'].value_counts()
artistas_validos = conteo_artistas[conteo_artistas >= umbral].index
df_filtrado = df_metadata[df_metadata['autor'].isin(artistas_validos)]

conteo_movimientos = df_filtrado['movimiento'].value_counts()
movimientos_validos = conteo_movimientos[conteo_movimientos >= umbral].index
df_filtrado = df_filtrado[df_filtrado['movimiento'].isin(movimientos_validos)]

conteo_artistas = df_filtrado['autor'].value_counts()
artistas_validos = conteo_artistas[conteo_artistas >= umbral].index
df_filtrado = df_filtrado[df_filtrado['autor'].isin(artistas_validos)]

df_filtrado

,id,autor,movimiento,genero
0,0,vincent-van-gogh,Realism,landscape
1,1,rembrandt,Baroque,religious_painting
2,2,paul-cezanne,Post_Impressionism,portrait
3,3,pierre-auguste-renoir,Impressionism,genre_painting
4,4,ivan-aivazovsky,Romanticism,Unknown Genre
...,...,...,...,...
67742,67742,sam-francis,Abstract_Expressionism,abstract_painting
67756,67756,sam-francis,Abstract_Expressionism,abstract_painting
67773,67773,sam-francis,Abstract_Expressionism,abstract_painting
67777,67777,sam-francis,Abstract_Expressionism,abstract_painting


In [109]:
dic_filtrado = df_filtrado.to_dict()

In [111]:

np.savez(os.path.join('DATA','metadata_filtrada'), data = dic_filtrado, allow_pickle=True)

# Extracción de características

In [ ]:
ids_errors = []
path_save = 'DATA/features'
ids = df_filtrado['id']

In [ ]:
q_min = -5
q_max = 5
s_min = 0.25
s_max = 0.75
s_degree = 2

dic_features = {'ids':[]}
save_every = 100
count_saves = 0

count = 0
for id in ids:
    item = ds[id]

    try:
        img = np.array(item['image'])
        img_norm = ut.normalize_image(img=img, max_size=1380, gray=True)
        for n_segments in range(1,4):
            list_images = ut.segment_image(img_norm, grid_size=n_segments)
            for p_segment,img_seg in enumerate(list_images):

                data_dfa, features_dfa = mf_dfa_features(img=img_seg, q_min=q_min, q_max=q_max, s_min=s_min,s_max=s_max, integration=True, degree_trend = 2, degree_scales = s_degree)
                data_renyi, features_renyi = mf_renyi_features(img=img_seg, q_min=q_min, q_max=q_max, s_min=s_min,s_max=s_max, degree_scales=s_degree)
                data_tda, features_tda = tda_features(img = img_seg, step=5)

                # Guarda características DFA
                for key in features_dfa.keys():
                    name_feature = f'{n_segments}__{p_segment}__' + key
                    if name_feature not in dic_features.keys():
                        dic_features[name_feature] = [features_dfa[key]]
                    else:
                        dic_features[name_feature].append(features_dfa[key])
                
                # Guarda características Renyi
                for key in features_renyi.keys():
                    name_feature = f'{n_segments}__{p_segment}__' + key
                    if name_feature not in dic_features.keys():
                        dic_features[name_feature] = [features_renyi[key]]
                    else:
                        dic_features[name_feature].append(features_renyi[key])

                # Guarda características TDA
                for key in features_tda.keys():
                    name_feature = f'{n_segments}__{p_segment}__' + key
                    if name_feature not in dic_features.keys():
                        dic_features[name_feature] = [features_tda[key]]
                    else:
                        dic_features[name_feature].append(features_tda[key])

        dic_features['ids'].append(id)

    except:
        ids_errors.append(id)
        
    count += 1
    if count % save_every == 0:
        count_saves += 1
        file_name = f'{count_saves}.pkl'
        with open(os.path.join(path_save,file_name), 'wb') as f:
            pickle.dump(dic_features, f)
        dic_features = {'ids':[]} # Reinicia diccionario
        print(f"Guardado {count_saves} de imagen {count}")
        # break
 

In [107]:
count_saves += 1
file_name = f'{count_saves}.pkl'
with open(os.path.join(path_save,file_name), 'wb') as f:
    pickle.dump(dic_features, f)

# No Suma

In [116]:
df = pd.DataFrame(np.load('DATA/features.npz', allow_pickle=True)['data'].item())

# Reemplazar inf y -inf por NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Eliminar filas con NaN
df = df.dropna()

# Eliminar duplicados
df = df.drop_duplicates()

# (Opcional) Resetear índice
df = df.reset_index(drop=True)


In [120]:
ids = df['ids']
ids_errors = []
path_save = 'DATA/features_nosum'


In [ ]:
q_min = -5
q_max = 5
s_min = 8
s_max = 0.25
s_degree = 2

dic_features = {'ids':[]}
save_every = 100
count_saves = 0

count = 0
for id in ids:
    item = ds[id]

    try:
        img = np.array(item['image'])
        img_norm = ut.normalize_image(img=img, max_size=1380, gray=True)
        for n_segments in range(1,4):
            list_images = ut.segment_image(img_norm, grid_size=n_segments)
            for p_segment,img_seg in enumerate(list_images):

                data_dfa, features_dfa = mf_dfa_features(img=img_seg, q_min=q_min, q_max=q_max, s_min=s_min,s_max=s_max, integration=False, degree_trend = 1, degree_scales = s_degree)
                data_renyi, features_renyi = mf_renyi_features(img=img_seg, q_min=q_min, q_max=q_max, s_min=s_min,s_max=s_max, degree_scales=s_degree)

                # Guarda características DFA
                for key in features_dfa.keys():
                    name_feature = f'{n_segments}__{p_segment}__' + key
                    if name_feature not in dic_features.keys():
                        dic_features[name_feature] = [features_dfa[key]]
                    else:
                        dic_features[name_feature].append(features_dfa[key])
                
                # Guarda características Renyi
                for key in features_renyi.keys():
                    name_feature = f'{n_segments}__{p_segment}__' + key
                    if name_feature not in dic_features.keys():
                        dic_features[name_feature] = [features_renyi[key]]
                    else:
                        dic_features[name_feature].append(features_renyi[key])


        dic_features['ids'].append(id)

    except:
        ids_errors.append(id)
        
    count += 1
    if count % save_every == 0:
        count_saves += 1
        file_name = f'{count_saves}.pkl'
        with open(os.path.join(path_save,file_name), 'wb') as f:
            pickle.dump(dic_features, f)
        dic_features = {'ids':[]} # Reinicia diccionario
        print(f"Guardado {count_saves} de imagen {count}")
        # break
 

In [122]:
count_saves += 1
file_name = f'{count_saves}.pkl'
with open(os.path.join(path_save,file_name), 'wb') as f:
    pickle.dump(dic_features, f)